# Неделя 3 — Baseline-модели

Задание: `docs/week3_baseline.md`

In [1]:
import sys
sys.path.append('..')
import warnings
import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import FunctionTransformer, RobustScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression

from src.validation import evaluate, time_split

dataset = pd.read_parquet('../data/processed/dataset.parquet')
CAT_COLS = ['segment', 'product', 'region']
NUM_COLS = [c for c in dataset.columns
            if c not in CAT_COLS + ['client_id', 'snapshot_date', 'target']]
print(len(NUM_COLS), 'числовых признаков')

18 числовых признаков


## 1. Разбиение по времени (out-of-time)

In [2]:
# train заканчивается на 2025-03 (прогноз последнего train-снапшота),
# test начинается с наблюдения 2025-05 -> пересечения месяцев нет
train, val, test = time_split(
    dataset,
    ['2024-07', '2024-10'],   # train
    ['2025-01'],              # val (для early stopping в 04)
    ['2025-10'],              # test
)

X_train, y_train = train[NUM_COLS + CAT_COLS], train['target']
X_test, y_test = test[NUM_COLS + CAT_COLS], test['target']
print('train:', train.shape, 'test:', test.shape)

train: (60191, 24) test: (36123, 24)


## 2. Baseline 0 — правило

In [3]:
m_rule = evaluate(y_test, -X_test['revenue_last_to_mean'], 'Правило: падение платежа')

--- Правило: падение платежа ---
ROC-AUC: 0.7934
PR-AUC: 0.3597
Precision@10%: 0.4596
Lift@10%: 5.66x
Доля оттока: 0.0812


## 3. Baseline 1 — логистическая регрессия

In [4]:
def signed_log1p(x):
    """Симметричный логарифм: выручка с тяжёлым хвостом -> нормальный масштаб."""
    return np.sign(x) * np.log1p(np.abs(x))

preprocess = ColumnTransformer([
    ('num', Pipeline([
        ('log', FunctionTransformer(signed_log1p, validate=False)),
        ('scale', RobustScaler()),
    ]), NUM_COLS),
    ('cat', OneHotEncoder(handle_unknown='ignore'), CAT_COLS),
])

model_lr = Pipeline([
    ('pre', preprocess),
    ('clf', LogisticRegression(max_iter=2000, C=0.1, class_weight='balanced')),
])

with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    model_lr.fit(X_train, y_train)

score_lr = model_lr.predict_proba(X_test)[:, 1]
m_lr = evaluate(y_test, score_lr, 'Логрегрессия')

--- Логрегрессия ---
ROC-AUC: 0.8270
PR-AUC: 0.4813
Precision@10%: 0.4898
Lift@10%: 6.03x
Доля оттока: 0.0812


/Users/viktorkovarov/Documents/b2b-churn-internship/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/viktorkovarov/Documents/b2b-churn-internship/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/viktorkovarov/Documents/b2b-churn-internship/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b


## 4. Сравнение baseline'ов


In [5]:
baseline_table = pd.DataFrame([m_rule, m_lr]).set_index('label')
print(baseline_table[['roc_auc', 'pr_auc', 'precision_at_10', 'lift_at_10']].round(4))

                          roc_auc  pr_auc  precision_at_10  lift_at_10
label                                                                 
Правило: падение платежа   0.7934  0.3597           0.4596      5.6583
Логрегрессия               0.8270  0.4813           0.4898      6.0298
